<a href="https://colab.research.google.com/github/hjiwoong/ML/blob/main/Chapter09_%EC%B6%94%EC%B2%9C%EC%8B%9C%EC%8A%A4%ED%85%9C_%EC%8B%A4%EC%8A%B5(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm # 컬러맵(colormap)
import matplotlib.font_manager as fm
import shutil, os

uploaded = files.upload()
font_dst = '/usr/local/share/fonts/NanumGothic.ttf'
os.makedirs('/usr/local/share/fonts', exist_ok=True)
shutil.copy('NanumGothic.ttf', font_dst)

fm.fontManager.ttflist = [
    f for f in fm.fontManager.ttflist
    if '/content/drive' not in f.fname
]

fm.fontManager.addfont(font_dst)
mpl.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots()
ax.set_title('한글 테스트')
ax.set_xlabel('테스트 축')
plt.show()

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### 잠재 요인 협업 필터링 - 경사 하강법 행렬 분해
행렬 분해(Matrix Factorization) 기반 추천 시스템 개념

[추천 시스템의 핵심 문제]
* 사용자-아이템 평점 행렬 R에서 대부분의 값이 NaN (미평가)
* NaN 위치의 평점을 예측해 높은 점수의 아이템을 추천

R ≈ P × Qᵀ
사용자 잠재 요인(P)과 아이템 잠재 요인(Q)으로 평점 행렬 분해

SGD로 P, Q를 반복 업데이트하여 예측 오류 최소화

[잠재 요인(Latent Factor)이란?]
* 명시적으로 정의되지 않은 숨겨진 특성
* 예: 영화의 경우 → 액션성, 로맨틱함, 코미디 요소 등
* 사용자는 자신의 취향(P), 아이템은 특성(Q)을 K차원으로 표현
* P × Qᵀ 내적 → 사용자-아이템 선호도 점수(예측 평점) 계산

[행렬 분해란?]
R ≈ P × Qᵀ
* R: (사용자 × 아이템) 원본 평점 행렬 (NaN 다수)
* P: (사용자 × K) 사용자 잠재 요인 행렬
* Q: (아이템 × K) 아이템 잠재 요인 행렬
* K: 잠재 요인(Latent Factor) 차원 수

In [ ]:
#원본 평점 행렬 R 생성, 행 : 사용자 4명, 열: 아이템5개, NaN : 미평가 항목 -> 이 값을 예측하는 것이 목표
R = np.array([[4, np.nan, np.nan, 2, np.nan],
              [np.nan, 5, np.nan, 3, 1],
              [np.nan, np.nan, 3, 4, 4],
              [5, 2, 1, 2, np.nan]])
num_users, num_items = R.shape#(4,5)
K = 3 #잠재 요인 차원 수(하이퍼파라미터), 너무 크면 : 과적합, 실제 Netflex prize 우승 모델 : k-200 이상 사용
np.random.seed(1)
P = np.random.normal(scale=1./K, size=(num_users, K))#정규분포(scale=1/K)로 작은 값으로 초기화
Q = np.random.normal(scale=1./K, size=(num_items, K))#너무 큰 초기값은 SGD 초기 단계에서 발산 위험
#SGD(확률적 경사 하강법)는 모델의 에측 오차를 줄이기 위해 P와 Q행렬을 조금씩(확률적으로)업데이트하는 최적화 방법
#목표는 실제 평점과 에측 평점 사이의 차이를 최소화하는 것
print('R shape:', R.shape)
print('P shape:', P.shape)
print('Q shape:', Q.shape)


In [ ]:
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
  full_pred_matrix = np.dot(P, Q.T)

  x_non_zero_ind = [nz[0] for nz in non_zeros] # 행(사용자) 인덱스
  y_non_zero_ind = [nz[1] for nz in non_zeros] # 열(아이템) 인덱스
  R_non_zeros = R[x_non_zero_ind, y_non_zero_ind] # 해당 위치의 실제값
  pred_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind] # 해당 위치의 예측값

  mse = mean_squared_error(R_non_zeros, pred_non_zeros)
  return np.sqrt(mse)

In [ ]:
# SGD(확률적 경사 하강법) 기반 행렬 분해
# [SGD 업데이트 원리]
# 목표: R ≈ P × Qᵀ 가 되도록 P, Q를 반복 업데이트

# NaN이 아닌 실제 평점 위치 추출
non_zeros = [(i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0]
non_zeros
# 이 위치들만 SGD 업데이트에 사용

steps         = 1000  # 전체 반복 횟수
learning_rate = 0.01  # 학습률
r_lambda      = 0.01  # L2 규제 강도

for step in range(steps):
  for i, j, r in non_zeros:
    eij = r - np.dot(P[i, :], Q[j, :].T) # i번 사용자와 j번 아이템의 내적 (예측 평점)
    # P, Q 동시 업데이트, Q[j]와 P[i] 값은 아직 업데이트되기 전의 이전 값을 사용
    P[i, :] += learning_rate * (eij * Q[j, :] - r_lambda * P[i, :])
    # 사용자 i의 선호도를 아이템 j의 특성(Q[j, :]) 방향으로 강화
    # 사용자 i의 잠재 요인 벡터(P[i, :])를, j번째 아이템에 대한 예측 평점의 오차(eij)를 줄이는  방향으로 조정
    Q[j, :] += learning_rate * (eij * P[i, :] - r_lambda * Q[j, :])
    # 아이템의 특성을 사용자의 실제 평점 패턴에 더 잘 맞도록 수정
    # j번쨰 아이템의 잠재 요인 벡터(Q[j, :])를, i번쨰 사용자가 해당 아이템에 대한 예측 평점의 오차 (eij)를 줄이는 방향으로 조정

  rmse = get_rmse(R, P, Q, non_zeros)
  if step % 50 == 0:
    print(f'step: {step:4d} RMSE: {rmse:.6f}')

In [ ]:
# 최종 예측 평점 행렬 생성
pred_matrix = np.dot(P, Q.T)

print('예측 평점 행렬 (NaN 위치도 예측됨):')
print(np.round(pred_matrix, 3))

print('\n원본 R (비교):')
print(np.round(R, 1))

### 콘텐츠 기반 필터링 - TMDB 5000 영화
영화 장르 속성으로 유사도 계산 -> 비슷한 장르 영화 추천
https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

In [ ]:
movies = pd.read_csv('/content/drive/MyDrive/KWU/ML/Data/tmdb_5000_movies.csv')
print('원본 데이터 shape:', movies.shape)

movies_df = movies[['id','title','genres','vote_average',
                    'vote_count','popularity','keywords','overview']]
pd.set_option('max_colwidth', 100)
print(movies_df[['genres','keywords']].iloc[0])
movies_df.head(3)


In [ ]:
movies_df.info()
# genros   [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]

In [ ]:
from ast import literal_eval
# 문자열로 저장된 파이썬 리스트/딕셔너리를 실제 파이썬 객체로 변환
movies_df['genres'] = movies_df['genres'].apply(literal_eval)
movies_df['keywords'] = movies_df['keywords'].apply(literal_eval)

# [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]
movies_df['genres'] = movies_df['genres'].apply(lambda x : [ y['name'] for y in x])
movies_df['keywords'] = movies_df['keywords'].apply(lambda x : [ y['name'] for y in x])
print('변환 후 genros (첫 번째 영화):', movies_df['genres'].iloc[0])
print('변환 후 keywords (첫 번째 영화):', movies_df['keywords'].iloc[0][:5])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# 장르 기반 콘텐츠 유사도 행렬 계산

movies_df['genros_literal'] = movies_df['genres'].apply(lambda x : (' ').join(x))

count_vect = CountVectorizer(
    min_df=1,
    ngram_range=(1,2)
)
genre_mat = count_vect.fit_transform(movies_df['genros_literal']) # 결과: (영화수 x 장르피처수) CSR 희소 행렬
print('장르 행렬 shape:',genre_mat.shape)

genre_sim = cosine_similarity(genre_mat, genre_mat) # 결과: (영화수 x 영화수) 정방 행렬
print('유사도 행렬 shape:', genre_sim.shape)

genre_sim_sorted_ind = genre_sim.argsort()[:, ::-1] #유사도 내림차순 정렬 인덱스

ironman_idx = movie_df[movies_df['title'] == 'Iron Man'].index.values[0]
top5_idx = genre_sim_sorted_ind[ironman_idx, 1:6] #i번쨰 영화가 가장 유사한 영화의 인덱스 1~5
print(movies_df.iloc[top5_idx]['title'].values)

In [ ]:
# 최종 추천 함수: 장르 유사도 + 가중 . 평점 . 결합
# 장르가 유사한 영화 중에서 가장 평점이 높은 것을 선별 -> 콘텐츠 유사도 + 품질 필터링 동사 적용


In [ ]:
# 단순 평균 평점
print('평균 평점 상위 10개 (vate_count 확인):')
print(movies_df[['title','vote_average','vote_count']]
      .sort_values('vote_average', ascending=False)[:10])

### IMDB 가중 평점(Weighted Rating) 공식
[IMDB 공식]

WR = (v / (v + m)) * R + (m / (v + m)) * C
* v: 해당 영화의 투표 수 (vote_count)
* m: 최소 기준 투표 수 (percentile로 결정)
* R: 해당 영화의 평균 평점 (vote_average)
* C: 전체 영화의 평균 평점

* v >> m (투표 많음): WR ≈ R (자체 평점이 지배)
* v << m (투표 적음): WR ≈ C (전체 평균으로 수렴, 신뢰도 보정)
* 투표 수가 적은 영화는 전체 평균으로 당겨짐, 극단적 평점 완화

In [ ]:
# IIMBD 가중 폄점

percentile = 0.6
m = movies_df['vote_count'].quantile(percentile) #상위 40% 영화만 신뢰도 있는 추천 대상
C = movies_df['vote_average'].mean() #전체 영화의 평균 평점
print(f'C (전체 평균 평점):[C:,3f]')
print(f'm (기존 투표 수):[m:,3f]')

def weighted_vote_average(record):
  v = record['vote_count']

  movies_df['weighted_vote'] = movies_df.apply (weighted_vote_average, axis=1) # 열방향 가로방향, 각 행(영화)에 함수 적용
  print('\n 가중 폄정 상위 10개:')
  print(movies_df[['title','vote_average','vote_count']]
        .sort_values('weighted_vote', ascending=False)[:10])

In [ ]:
# 최종 추천 함수: 장르 유사도 + 가중 평점 결합
# 장르가 유사한 영화 중에서 가중 평점이 높은 것을 선별 -> 콘텐츠 유사도 + 품질 필터링 동사 적용
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
  title_movie = df[df['title'] == title_name]
  title_index = title_movie.index.values

  similar_indexes = sorted_ind[title_index, :(top_n * 2)] #유사도 상위 top_n*2개 추출
  similar_indexes = similar_indexes.reshape(-1) # 20 -> 10로 평탄화

  similar_indexes = similar_indexes[similar_indexes != title_index] #자기 자신 제외
  return df.iloc[similar_indexes].sort_values('weighted_vote', ascending=False)[:top_n]

similar_movies = find_sim_movie(movies_df, genre_sim_sorted_ind, 'Iron man', 10)
print(' iron man 유사한 영화 (장르 유사도 + 가중평점) ')
print(similar_movies[['title','vote_average','weighted_vote']].to_string(index=False))

### 아이템 기반 협업 필터링(Item-Based Collaborative Filtering)

* "나와 비슷한 취향의 사람들이 좋아한 것을 나도 좋아할 것"
* 사용자들의 과거 평점 행동 패턴에서 유사성을 찾아 추천

[사용자 기반 vs 아이템 기반]
* 사용자 기반: 비슷한 취향의 사용자들이 본 영화 추천
* 아이템 기반: 내가 좋아한 영화와 유사한 영화 추천

[MovieLens 데이터셋]
https://grouplens.org/datasets/movielens/latest/
* GroupLens Research의 영화 평점 데이터 (비공개 연구용)
* ml-latest-small: 약 100,000개 평점, 9,000개 영화, 600명 사용자GroupLensMovieLens Latest DatasetsThese datasets will change over time, and are not appropriate for reporting research results. We will keep the download links stable for automated downloads. We will not archive or make available p…2015년 9월 24일

In [ ]:
movies = pd.read_csv('/content/drive/MyDrive/KWU/ml-latest-small/movies.csv')
rating = pd.read_csv('/content/drive/MyDrive/KWU/ml-latest-small/ratings.csv')
print('movies shape;', movies,shape) #(영화수, 컬럼수)
print('ratings shape;', ratings,shape) #(평점수, 컬럼수)
display(movies.head(3))
display(ratings.head(3))

In [ ]:
#사용자 아이템 평점 행렬 생성
ratings_m = ratings[['userID','movieID','rating']]
rating_movies = pd.merge(ratings_m, movies, on='movieID')
rating_movies.head(3)

In [ ]:
rating_matrix = rating_movies.pivot_table('rating', index= 'userid', columns='title') #DataFrame의 데이터를 요약하고
rating_matrix.head(3) # 사용자- 영화 평점 행렬 생성

In [ ]:
ratings_matrix = rating_matrix.fillna(0) #NaN은 0으로 대체
print('사용자-영화 평점 행렬 shape:',ratings_matrix.shape)
ratings_matrix.iloc(:3, :5)

In [ ]:
# 아이템-아이템 코사인 유사도 행렬 계산
# [아이템 유사도 계산 원리]
# 각 영화를 '해당 영화를 본 모든 사용자들의 평점 백터'를 표현
# 두 영화의 평점 패턴이 비슷하면 > 코사인 유사도 높음
# cosine_similartity는 행 단위로 유사도계산, 영화는 전차가 필요
ratings_matrix_T = ratings_matrix.transpose() #전치
print('전치 행렬 shape:', ratings_matrix_T.shape)
display(ratings_matrix_T.head(3)) #영화 x 사용자

In [ ]:
# 영화 x 영화 코사인 유사도 행렬
item_sim = cosine__similarity(ratings_matrix_T, ratings_matrix_T)
item_sim_df = pd.DataFrame(data=item_sim, index=ratings_matrix.columns, columns=ratings_matrix.columns)

In [ ]:
# 영화 X 영화 코사인 유사도 행렬
item_sim = cosine_similarity(ratings_matrix_T, ratings_matrix_T)

item_sim_df = pd.DataFrame
 (data=item_sim,
  index=ratings_matrix.columns, # 행: 영화 제목
  columns=ratings_matrix.columns # 열: 영화 제목
  )
 print('유사도 행렬 shape:', item_sim_df.shape)
 print('\nIron Man (2008)와 유사항 상위 5편:')
 print(item_sim_df['iron Man (2008)'].sort_values(ascending=False)[:6])

### 예측 평점 계산: 전체 이웃 방식
[아이템 기반 예측 평점 공식]\
사용자 u의 아이템 i에 대한 예측 평점:

r̂_ui = Σ(sim(i,j) × r_uj) / Σ|sim(i,j)|

(j: u가 평점을 준 모든 아이템)
* 사용자가 유사한 아이템들에 준 평점의 가중 평균, 유사도가 높은 아이템의 평점이 더 큰 영향을 미침

In [ ]:
# 예측 평점 계산: 전체 이웃 방식
def predict_rating(ratings_arr, item_sim_arr): #모든 아이템 이웃을 사용한 예측 평점 계산
  ratings_pred = ratings_arr.dot(item_sim_arr) / np.array([np.abs(item_sim_arr).sum(axis=1)])
  return ratings_pred
  # 분자 : (사용자 x 아이템) X (아이템 x 아이템) = (사용자 x 아이템)
  # 분모 ; 각 영화가 다른 모든 영화들과 가지는 유사도의 '절대적인 합계'를 구해서 '평균'을 내거나 '정규화' 하는 데 사용
  # 특정 영화가 다른 영화들과의 유사도 높을수록 예측 평점에 더 큰 가중치를 주는 식으로 작동

def get_mse(pred, actual): #실제 평점이 있는 위치 (nonzero)만 선택하여 MSE 계산
  pred = pred[actual.nonzero()].flatten
  actual = actual[actual.nonzero()].flatten()
  return mean_squared_error(pred, actual)

ratings_pred =  predict_rating(ratings_matrix.values, item_sim_df.values)# rating_matrix 사용자-영화 평점 행렬 , 유사
print('아이템 기반 전체 이웃 MSE:' , get_mse(ratings_pred, ratings_matrix.values))
#MSE(Mean Squared Error, 평균 제곱 오차)는 값이 낮을수록 좋다.

In [ ]:
# 예측 평점 계산: Top-N 이웃 방식 (성능 향상)
# 전체 이웃 방식: 유사도가 매우 낮은 관련 없는 아이템도 포함
# Top-N 이웃 방식: 가장 유사한 N개만 선택 > 더 정확한 예측, 계산량도 감소, 노이즈 제거
Def predict_rating_topsim(ratings_arr, item_sim_arr, n=20):
  pred = np.zeros(ratings_arr.shape)
  for col in range(ratings_arr.shape[1]):#모든 영화를 하나씩 순회하면서, 현재 영화(col)에 대한 예측 평점을 계산
  top_n_items = [np,argosrt(item_sim_arr[;, col])[:-n-1;-1]] # 현재 영화(col)과 유사도가 높은 상위 n개 아이템 인덱스
  for row in range(ratings_arr.shape[0]): # 선택된 현재 영화에 대해, 모든 사용자의 평점을 하나씩 에측하는 작업
    pred [row, col] = (
        item_sim_arr[top_n_item,col] * ratings_arr[row, top_n_items]).sum() #row번 사용자가 가장 비슷한 N개의 영화들에 실제로 준 평점 리스트
        / np.sum(np.abs(item_sim_arr[top_n_items, col])) #분모: Top-N 유사도 절대값 합 (정규화)
    return pred


ratings_pred = predict_rating_topsim(ratings_matrix.values, item_sim_df.values, n=20)
print('아이템 기반 Top-N 이웃 MSE:' , get_mse(ratings_pred, ratings_matrix.values))

In [ ]:
ratings_pred_matrix = pd.DataFrame(
    data=ratings_pred_top20,
  index=ratings_matrix.index, #행: userid
  columns=ratings_matrix.columns # 열: 영화제목
)

In [ ]:
# 개인화 추천: 미시청 영화 필터링 + 예측 평점 순 추천
# [실제 서비스 추천 흐름]
# 1. 사용자가 이미 본 영화 파악
# 2. 보지 않은 영화 목록 추출 (미시청 = 추천 대상)
# 3. 예측 평점이 높은 순서로 정렬
# 4. 상위 n 판 추